In [1]:
from model.nicheDeconv import *
import pickle

In [2]:
type_list = ['stroma',
 'plasmablast-enriched stroma',
 'lymphoid structure',
 'tumor-stroma boundary',
 'myeloid-enriched stroma',
 'tumor interior']

In [3]:
with open(f'../../data/human_nsclc/nsclc_norm', 'rb') as f:
    train_x_pack = pickle.load(f)
    test_x_pack = pickle.load(f)

In [4]:
train_x_sim_list, train_y_list = train_x_pack
test_x_sim_list, test_y_list = test_x_pack

In [5]:
valid_size = 1000  
# 切片操作  
valid_x_sim_list = train_x_sim_list[:valid_size]  
valid_y_list = train_y_list[:valid_size]  

train_x_sim_list = train_x_sim_list[valid_size:]  
train_y_list = train_y_list[valid_size:] 

In [6]:
# 
with open("../../data/human_nsclc/sc_gene_names.pkl", "rb") as f:
    sc_gene_names = pickle.load(f)
sc_spatial_var_genes = pd.read_csv('../../data/human_nsclc/nsclc_spatial_var_gene.csv')
sc_spatial_var_genes['g'] = pd.Categorical(sc_spatial_var_genes['g'], categories=sc_gene_names, ordered=True)
sc_spatial_var_genes = sc_spatial_var_genes.sort_values('g')
sc_spatial_var_genes = sc_spatial_var_genes.reset_index(drop=True)
features_names = ['FSV', 'max_delta', 'max_ll', 'max_mu_hat', 'max_s2_t_hat', 's2_FSV', 's2_logdelta', 'LLR', 'pval', 'qval']
sc_spatial_var_genes = sc_spatial_var_genes[features_names]
scaled_df, log_cols = auto_log_then_minmax(sc_spatial_var_genes)
scaled_np = scaled_df.values.flatten()
scaled_np.shape

(9600,)

In [7]:
valid_x_sim = merge_full_array_to_series_list(valid_x_sim_list, scaled_np)
train_x_sim = merge_full_array_to_series_list(train_x_sim_list, scaled_np)
test_x_sim = merge_full_array_to_series_list(test_x_sim_list, scaled_np)

source_data = data2h5ad(train_x_sim, train_y_list, type_list)
target_data = data2h5ad(test_x_sim, test_y_list, type_list)
valid_data = data2h5ad(valid_x_sim, valid_y_list, type_list)

AnnData object with n_obs × n_vars = 13984 × 10560
    obs: 'stroma', 'plasmablast-enriched stroma', 'lymphoid structure', 'tumor-stroma boundary', 'myeloid-enriched stroma', 'tumor interior'
    uns: 'cell_types'
AnnData object with n_obs × n_vars = 1215 × 10560
    obs: 'stroma', 'plasmablast-enriched stroma', 'lymphoid structure', 'tumor-stroma boundary', 'myeloid-enriched stroma', 'tumor interior'
    uns: 'cell_types'
AnnData object with n_obs × n_vars = 1000 × 10560
    obs: 'stroma', 'plasmablast-enriched stroma', 'lymphoid structure', 'tumor-stroma boundary', 'myeloid-enriched stroma', 'tumor interior'
    uns: 'cell_types'


In [8]:
with open("../../data/human_nsclc/nsclc_906.pkl", "rb") as f:
    graph = pickle.load(f)
with open("../../data/human_nsclc/gcn_gene_names.pkl", "rb") as f:
    gcn_gene_names = pickle.load(f)
with open("../../data/human_nsclc/sc_gene_names.pkl", "rb") as f:
    sc_gene_names = pickle.load(f)

In [9]:
gene_indices = [sc_gene_names.index(gene) for gene in gcn_gene_names]

In [10]:
expression_data = source_data.X[:, gene_indices]  
source_data.obsm['new_gene_expression'] = expression_data
expression_data = target_data.X[:, gene_indices] 
target_data.obsm['new_gene_expression'] = expression_data
expression_data = valid_data.X[:, gene_indices] 
valid_data.obsm['new_gene_expression'] = expression_data

In [11]:
gene2idx = {g: i for i, g in enumerate(gcn_gene_names)}
num_nodes = len(gcn_gene_names)

In [12]:
edge_index_list = []
edge_weight_list = []

for u, v, data in graph.edges(data=True):
    i = gene2idx[u]
    j = gene2idx[v]
    
    w = data["weight"]   
    
    # u -> v
    edge_index_list.append([i, j])
    edge_weight_list.append(w)
    # v -> u
    edge_index_list.append([j, i])
    edge_weight_list.append(w)

edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()   # [2, E]
edge_weight = torch.tensor(edge_weight_list, dtype=torch.float)    

In [13]:
model = nicheDeconv(200, 50, 0.0001)
model.register_graph(edge_index, edge_weight, len(gcn_gene_names))
patience = 10
pred_loss, best_model_weights = model.train(source_data, target_data, valid_data, patience, 'human_nsclc')

In [14]:
model.encoder.load_state_dict(best_model_weights['encoder'])
model.gnn.load_state_dict(best_model_weights['gnn'])
model.predictor.load_state_dict(best_model_weights['predictor'])

<All keys matched successfully>

In [15]:
target_preds, ground_truth = model.prediction(model.test_target_loader)
CCC, RMSE, Corr = compute_metrics(target_preds, ground_truth)
CCC, RMSE, Corr

(0.8629982092376902, 0.07586308698578724, 0.8715985869194725)

In [16]:
ckpt = {
    "encoder": model.encoder.state_dict(),
    "gnn": model.gnn.state_dict(),
    "predictor": model.predictor.state_dict(),
}

torch.save(ckpt, "../../save_models/human_nsclc/best_model.pt")  # 或 .pth

In [17]:
target_preds.to_csv(f'../../res/human_nsclc/nicheDeconv.csv')
ground_truth.to_csv(f'../../res/human_nsclc/real_ncslc.csv')